# **Data Cleaning**
___

In [1]:
import sys
from pathlib import Path
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

USE_MOCK_DATA = True

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import hashlib

from src.config import load_config
from src.paths import get_paths
from src.visualization import set_publication_style, save_figure

cfg   = load_config()
paths = get_paths()
set_publication_style()

# Load cleaned manifest from notebook 02
manifest_path = paths["interim"] / "manifest.csv"
manifest = pd.read_csv(manifest_path)
print(f"Loaded manifest: {manifest.shape[0]} rows, {manifest.shape[1]} columns")
print(manifest.dtypes)

Loaded manifest: 800 rows, 4 columns
image_path           object
label                 int64
dataset              object
original_filename    object
dtype: object


## Check 1: Label Validity

In [2]:
# Labels must be 0 or 1 — nothing else
valid_labels = manifest["label"].isin([0, 1])
invalid_rows = manifest[~valid_labels]
print(f"Rows with invalid labels: {len(invalid_rows)}")
if len(invalid_rows) > 0:
    print(invalid_rows)

# I keep only valid labels
manifest = manifest[valid_labels].reset_index(drop=True)
print(f"After filtering: {len(manifest)} rows remain")

Rows with invalid labels: 0
After filtering: 800 rows remain


## Check 2: File Existence

In [3]:
# Every image file listed in the manifest must actually exist on disk
missing_files = []
for idx, row in manifest.iterrows():
    if not Path(row["image_path"]).exists():
        missing_files.append(idx)

print(f"Missing image files: {len(missing_files)}")
if missing_files:
    print("Missing file paths:")
    for idx in missing_files[:5]:
        print(f"  {manifest.loc[idx, 'image_path']}")
    manifest = manifest.drop(index=missing_files).reset_index(drop=True)
    print(f"Removed {len(missing_files)} missing-file rows.")

print(f"Remaining: {len(manifest)} rows")

Missing image files: 0
Remaining: 800 rows


## Check 3: Image Loadability (Corruption Check)

In [4]:
# I try to open every image. If PIL raises an error, the file is corrupted.
corrupt_indices = []

for idx, row in manifest.iterrows():
    try:
        img = Image.open(row["image_path"])
        img.verify()    # Checks file integrity without fully decoding
    except Exception as e:
        corrupt_indices.append(idx)
        print(f"  Corrupted file at index {idx}: {row['image_path']} — {e}")

print(f"Corrupted images found: {len(corrupt_indices)}")
manifest = manifest.drop(index=corrupt_indices).reset_index(drop=True)
print(f"Remaining: {len(manifest)} rows")

Corrupted images found: 0
Remaining: 800 rows


## Check 4: Duplicate Detection via Image Hash

In [5]:
# Two images with identical pixel content get the same MD5 hash.
# Duplicates that appear in both train and test sets cause data leakage.

def image_md5(path):
    """Compute MD5 hash of image pixel data (not the file itself)."""
    try:
        img = Image.open(path).convert("L")  # Greyscale for consistency
        return hashlib.md5(img.tobytes()).hexdigest()
    except Exception:
        return None

print("Computing image hashes (may take a moment)...")
manifest["image_hash"] = manifest["image_path"].apply(image_md5)

# Find duplicates
duplicated_mask = manifest["image_hash"].duplicated(keep="first")
n_duplicates = duplicated_mask.sum()
print(f"Duplicate images (identical pixel content): {n_duplicates}")

if n_duplicates > 0:
    print("Duplicate examples:")
    dup_hashes = manifest.loc[duplicated_mask, "image_hash"].values
    print(manifest[manifest["image_hash"].isin(dup_hashes)][["image_path","label","image_hash"]].head(10))
    manifest = manifest[~duplicated_mask].reset_index(drop=True)
    print(f"After deduplication: {len(manifest)} rows")

Computing image hashes (may take a moment)...
Duplicate images (identical pixel content): 0


## Check 5: Near-Zero Variance (Blank or Overexposed Images)

In [6]:
# A chest X-ray with near-zero pixel variance is almost certainly blank or
# corrupted (e.g., all-black or all-white). These should be excluded because
# they carry no diagnostic information and can inflate model confidence.

LOW_VARIANCE_THRESHOLD = 100   # Pixel variance below this → suspicious

low_var_indices = []

for idx, row in manifest.iterrows():
    try:
        img_array = np.array(Image.open(row["image_path"]).convert("L"))
        variance = img_array.var()
        if variance < LOW_VARIANCE_THRESHOLD:
            low_var_indices.append(idx)
            print(f"  Low variance ({variance:.1f}) at index {idx}: {row['image_path']}")
    except Exception:
        pass

print(f"Near-zero variance images: {len(low_var_indices)}")
manifest = manifest.drop(index=low_var_indices).reset_index(drop=True)
print(f"Remaining: {len(manifest)} rows")

Near-zero variance images: 0
Remaining: 800 rows


## Check 6: Aspect Ratio Extremes

In [7]:
# Very extreme aspect ratios (e.g., 10:1) suggest the image is a strip
# or thumbnail, not a real CXR. I flag these for review.

ASPECT_RATIO_MAX = 3.0   # Width/Height or Height/Width

extreme_ar = []

for idx, row in manifest.iterrows():
    try:
        img = Image.open(row["image_path"])
        w, h = img.size
        ar = max(w/h, h/w)
        if ar > ASPECT_RATIO_MAX:
            extreme_ar.append(idx)
    except Exception:
        pass

print(f"Extreme aspect ratio images (>{ASPECT_RATIO_MAX}:1): {len(extreme_ar)}")
if extreme_ar:
    for idx in extreme_ar[:3]:
        img = Image.open(manifest.loc[idx, "image_path"])
        print(f"  Size: {img.size} → aspect ratio {max(img.size)/min(img.size):.2f}:1")
    # I keep these but flag them — CXRs can sometimes be wide-format
    manifest["extreme_aspect_ratio"] = manifest.index.isin(extreme_ar)

Extreme aspect ratio images (>3.0:1): 0


## Cleaning Summary Report

In [8]:
cleaning_report = {
    "Final dataset size": len(manifest),
    "TB Positive (label=1)": int((manifest["label"] == 1).sum()),
    "TB Negative (label=0)": int((manifest["label"] == 0).sum()),
    "Positive rate (%)": round(100 * (manifest["label"] == 1).mean(), 2),
}

print("Data cleaning complete. Summary:")
for key, val in cleaning_report.items():
    print(f"  {key}: {val}")

# Save cleaning report
report_df = pd.DataFrame([cleaning_report])
report_path = paths["tables"] / "data_cleaning_report.csv"
report_df.to_csv(report_path, index=False)
print(f"\nCleaning report saved: {report_path}")

# Save the cleaned manifest
clean_manifest_path = paths["interim"] / "manifest_clean.csv"
manifest.to_csv(clean_manifest_path, index=False)
print(f"Cleaned manifest saved: {clean_manifest_path}")

Data cleaning complete. Summary:
  Final dataset size: 800
  TB Positive (label=1): 394
  TB Negative (label=0): 406
  Positive rate (%): 49.25

Cleaning report saved: C:\Users\Peter\Documents\projects\Data_science\fedtb_nigeria\reports\tables\data_cleaning_report.csv
Cleaned manifest saved: C:\Users\Peter\Documents\projects\Data_science\fedtb_nigeria\data\interim\manifest_clean.csv
